In [1]:
import torch
import torch.nn as nn

def count_model_params(model: nn.Module) -> dict:
    """Computes trainable, non-trainable, and total parameters in PyTorch."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    return {
        "trainable": trainable,
        "non_trainable": non_trainable,
        "total": trainable + non_trainable,
    }

def get_conv1d_params(in_length: int, out_length: int, in_channels: int, out_channels: int, bias: bool = True):
    """
    Finds valid (kernel_size, stride, padding) configurations to map in_length -> out_length,
    along with total learnable parameters and an instantiated nn.Conv1d layer.
    """
    candidates = []
    for stride in [1, 2, 4, 8]:
        for kernel_size in range(2, 17):
            for padding in range(kernel_size):
                num = in_length + 2 * padding - kernel_size
                if num >= 0 and num % stride == 0:
                    l_out = num // stride + 1
                    if l_out == out_length:
                        num_params = out_channels * in_channels * kernel_size + (out_channels if bias else 0)
                        candidates.append({
                            "kernel_size": kernel_size,
                            "stride": stride,
                            "padding": padding,
                            "trainable_params": num_params,
                            "layer": nn.Conv1d(
                                in_channels=in_channels,
                                out_channels=out_channels,
                                kernel_size=kernel_size,
                                stride=stride,
                                padding=padding,
                                bias=bias,
                            ),
                        })
    return candidates


def get_conv_transpose1d_params(
    in_length: int,
    out_length: int,
    in_channels: int,
    out_channels: int,
    bias: bool = True,
):
    """
    Finds valid (kernel_size, stride, padding) configurations to map in_length -> out_length,
    along with total learnable parameters and an instantiated nn.ConvTranspose1d layer.
    """
    candidates = []
    for stride in [1, 2, 4, 8]:
        for kernel_size in range(2, 17):
            for padding in range(kernel_size):
                l_out = (in_length - 1) * stride - 2 * padding + kernel_size
                if l_out == out_length:
                    num_params = in_channels * out_channels * kernel_size + (out_channels if bias else 0)
                    candidates.append({
                        "kernel_size": kernel_size,
                        "stride": stride,
                        "padding": padding,
                        "trainable_params": num_params,
                        "layer": nn.ConvTranspose1d(
                            in_channels=in_channels,
                            out_channels=out_channels,
                            kernel_size=kernel_size,
                            stride=stride,
                            padding=padding,
                            bias=bias,
                        ),
                    })
    return candidates



In [2]:
configs = get_conv1d_params(in_length=2048, out_length=2048, in_channels=8, out_channels=1)
for c in configs:
    print(f"k={c['kernel_size']}, s={c['stride']}, p={c['padding']} | Params: {c['trainable_params']}")
# Pick the standard downsampling layer (k=4, s=2, p=1)


k=3, s=1, p=1 | Params: 25
k=5, s=1, p=2 | Params: 41
k=7, s=1, p=3 | Params: 57
k=9, s=1, p=4 | Params: 73
k=11, s=1, p=5 | Params: 89
k=13, s=1, p=6 | Params: 105
k=15, s=1, p=7 | Params: 121


In [3]:
import h5py

with h5py.File("nonhyb_noprec_dataset_q8.hdf", "r") as f:
    print("Keys:", list(f.keys()))  # ['Parameters', 'Waveforms']
    waveforms = f["Waveforms"][:]
    params = f["Parameters"][:]


Keys: ['Parameters', 'Waveforms']


In [4]:
print(params.shape)
print(waveforms.shape)

(102400, 7)
(102400, 2048)


In [5]:
configs = get_conv_transpose1d_params(in_length=2048, out_length=2048, in_channels=8, out_channels=1)
for c in configs:
    print(f"k={c['kernel_size']}, s={c['stride']}, p={c['padding']} | Params: {c['trainable_params']}")


k=3, s=1, p=1 | Params: 25
k=5, s=1, p=2 | Params: 41
k=7, s=1, p=3 | Params: 57
k=9, s=1, p=4 | Params: 73
k=11, s=1, p=5 | Params: 89
k=13, s=1, p=6 | Params: 105
k=15, s=1, p=7 | Params: 121


In [6]:
def split_data(X, Y, test_size=0.1, val_size=0.1, shuffle=True, random_state=42):
    """Splits the dataset into training, validation, and testing sets."""
    X_train_val, X_test, y_train_val, y_test = train_test_split(X, Y, test_size=test_size, random_state=random_state, shuffle=shuffle)
    X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=val_size, random_state=random_state, shuffle=shuffle)
    return X_train, X_val, X_test, y_train, y_val, y_test

def get_phase(waveform):
    phase_non = -1*np.unwrap(np.angle(waveform))/100
    out = phase_non
    phase= out-out[..., np.newaxis,0]
    return phase


(
    X_train_app_phase,
    X_val_app_phase,
    X_test_app_phase,
    y_train_app_phase,
    y_val_app_phase,
    y_test_app_phase,
) = split_data(params_app, get_phase(waves_app))

(
    X_train_app_ampl,
    X_val_app_ampl,
    X_test_app_ampl,
    y_train_app_ampl,
    y_val_app_ampl,
    y_test_app_ampl,
) = split_data(params_app, abs(waves_app))

NameError: name 'params_app' is not defined

# Dataset preparation / data loading etcc

In [ ]:
from torch.utils.data import DataLoader
from utils import separate_phase_ampl
from torch import Tensor
from torch.utils.data import random_split, TensorDataset
import h5py
import numpy as np
from utils import 


with h5py.File("nonhyb_noprec_dataset_q8.hdf", "r") as f:
    print("Keys:", list(f.keys()))  # ['Parameters', 'Waveforms']
    waveforms = f["Waveforms"][:100] #numpy.ndarray
    params = f["Parameters"][:100] # same prob


splitted_waveforms = separate_phase_ampl(waveforms)

a_train, a_val, a_test = random_split(splitted_waveforms[0], [0.8,0.1,0.1], generator=torch.Generator().manual_seed(42))
p_train, p_val, p_test = random_split(splitted_waveforms[1], [0.8,0.1,0.1], generator=torch.Generator().manual_seed(42))


a_train_loader = DataLoader(a_train, batch_size = 128, shuffle= True)
a_val_loader = DataLoader(a_val, batch_size=256, shuffle=False)
a_test_loader = DataLoader(a_test, batch_size=256, shuffle=False)
p_train_loader = DataLoader(p_train, batch_size = 128, shuffle= True)
p_val_loader = DataLoader(p_val, batch_size=256, shuffle=False)
p_test_loader = DataLoader(p_test, batch_size=256, shuffle=False)

Keys: ['Parameters', 'Waveforms']


numpy.ndarray

# Loss